In [ ]:
# Demo
# based on https://github.com/HTuennermann/Vehicle-Detection-and-Tracking (ported to PyTorch)

# Overview
- Simple object detector
- Input: RGB image
- Output: cars' locations

# Import libraries

In [ ]:
import glob
from collections import OrderedDict
from skimage.io import imread
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pylab as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# (1) Data

#### Car images: https://drive.google.com/file/d/1rqitUFrFhB84tFVtg1Ad1TVz6coCWpNZ/view?usp=sharing

In [ ]:
cars = glob.glob("./data/obj_detection/vehicles/*/*.png")
non_cars = glob.glob("./data/obj_detection/non-vehicles/*/*.png")

Y = np.concatenate([np.ones(len(cars)), np.zeros(len(non_cars)) - 1])
X = []
for name in cars:    
    X.append(imread(name))
for name in non_cars:    
    X.append(imread(name))
X = np.array(X)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30)

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

# (2) Declare model

In [ ]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class Lambda(nn.Module):
    """Equivalent of keras.layers.Lambda: wraps a function as a layer."""
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x):
        return self.fn(x)


def get_conv(input_shape=(3, 64, 64), filename=None):
    # input_shape (C, H, W) is informative only: the network is fully convolutional,
    # so it accepts any image with H, W >= 64
    model = nn.Sequential(OrderedDict([
        ('lambda', Lambda(lambda x: x/127.5 - 1.)),
        ('conv1', nn.Conv2d(input_shape[0], 10, (3, 3), padding=1)),  # padding="same"
        ('conv1_relu', nn.ReLU()),
        ('conv2', nn.Conv2d(10, 10, (3, 3), padding=1)),
        ('conv2_relu', nn.ReLU()),
        ('pool', nn.MaxPool2d(kernel_size=(8, 8))),
        ('dropout1', nn.Dropout(0.25)),
        ('dense1', nn.Conv2d(10, 128, (8, 8))),  # padding="valid"
        ('dense1_relu', nn.ReLU()),
        ('dropout2', nn.Dropout(0.5)),
        ('dense2', nn.Conv2d(128, 1, (1, 1))),
        ('dense2_tanh', nn.Tanh()),
    ]))
    model.apply(keras_init)
    if filename:
        model.load_state_dict(torch.load(filename, map_location=device))
    return model.to(device)


model = get_conv()
model.add_module('flatten', nn.Flatten())

print(model)
print('Total params:', sum(p.numel() for p in model.parameters()))
print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

# (3) Train defined model

In [ ]:
batch_size = 128

# PyTorch: images NHWC -> NCHW, labels (+1 car / -1 non-car) as (N, 1) to match the flattened output
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train.transpose(0, 3, 1, 2).copy()),
                                        torch.from_numpy(Y_train.astype('float32')).view(-1, 1)),
                          batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test.transpose(0, 3, 1, 2).copy()),
                                       torch.from_numpy(Y_test.astype('float32')).view(-1, 1)),
                         batch_size=batch_size, shuffle=False)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Keras 'adam' default lr


def binary_accuracy(out, y):
    # same as Keras 'accuracy' for a single output unit: round(prediction) == label
    return (torch.round(out) == y).float().mean()


def run_epoch(model, loader, criterion, acc_fn, optimizer=None):
    """One pass over `loader`; trains if an optimizer is given, otherwise evaluates."""
    train = optimizer is not None
    model.train(train)
    total_loss, total_acc, n = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            bs = xb.size(0)
            total_loss += loss.item() * bs
            total_acc += acc_fn(out, yb).item() * bs
            n += bs
    return total_loss / n, total_acc / n


def fit(model, train_loader, val_loader, criterion, optimizer, epochs, acc_fn):
    """Replacement for Keras model.fit: returns a history dict."""
    history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        loss, acc = run_epoch(model, train_loader, criterion, acc_fn, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, acc_fn)
        for k, v in zip(history, (loss, acc, val_loss, val_acc)):
            history[k].append(v)
        print(f'Epoch {epoch + 1}/{epochs} - loss: {loss:.4f} - acc: {acc:.4f}'
              f' - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')
    return history


def evaluate(model, loader, criterion, acc_fn):
    """Replacement for Keras model.evaluate: returns [loss, accuracy]."""
    return list(run_epoch(model, loader, criterion, acc_fn))


history = fit(model, train_loader, test_loader, criterion, optimizer, epochs=5, acc_fn=binary_accuracy)
score = evaluate(model, test_loader, criterion, binary_accuracy)
print('Test score:', score[0])
print('Test accuracy:', score[1])
torch.save(model.state_dict(), "localize.pt")

# (4) Evaluate trained model

In [ ]:
heatmodel = get_conv(input_shape=(3, None, None), filename="localize.pt")

In [ ]:
def locate():
    data = imread("test.jpg")
    
    heatmodel.eval()
    with torch.no_grad():
        inp = torch.from_numpy(data.transpose(2, 0, 1)[None].astype(np.float32)).to(device)  # (1, 3, H, W)
        heatmap = heatmodel(inp).cpu().numpy().transpose(0, 2, 3, 1)  # NCHW -> NHWC: (1, H', W', 1)
    sel_heatmap = heatmap[0, :, :, 0] > 0.99

    xx, yy = np.meshgrid(np.arange(heatmap.shape[2]), np.arange(heatmap.shape[1]))
    x = (xx[heatmap[0, :, :, 0] > 0.99])
    y = (yy[heatmap[0, :, :, 0] > 0.99])
    
    for i, j in zip(x, y):
        data[j*8:j*8+64, i*8] = (0, 255, 0)
        data[j*8:j*8+64, i*8+64] = (0, 255, 0)
        
        data[j*8, i*8:i*8+64] = (0, 255, 0)
        data[j*8+64, i*8:i*8+64] = (0, 255, 0)
    return heatmap[0, :, :, 0], sel_heatmap, data

hm, sel_hm, annotated = locate()

plt.figure(figsize=(10, 20))

plt.subplot(3, 1, 1)
plt.title("Heatmap")
plt.imshow(hm)

plt.subplot(3, 1, 2)
plt.title("Car Area")
plt.imshow(sel_hm)

plt.subplot(3, 1, 3)
plt.title("Detected Cars")
plt.imshow(annotated)
plt.show()